In [0]:
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DoubleType
)

from pyspark.sql.functions import (
    col,
    current_timestamp,
    to_timestamp
)

# Storage paths
base_stream_path = (
    "/Volumes/fraud_detection/bronze/realtime_files"
)

incoming_path = (
    f"{base_stream_path}/incoming_transactions"
)

schema_path = (
    f"{base_stream_path}/schema"
)

checkpoint_path = (
    f"{base_stream_path}/checkpoints/bronze_transactions"
)

# Explicit incoming transaction schema
transaction_schema = StructType([
    StructField("transaction_id", StringType(), False),
    StructField("card_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("merchant_id", StringType(), True),
    StructField("amount", DoubleType(), True),
    StructField("timestamp", StringType(), True)
])

# Read new JSON files with Auto Loader
bronze_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", schema_path)
    .option("rescuedDataColumn", "_rescued_data")
    .schema(transaction_schema)
    .load(incoming_path)
    .withColumn(
        "transaction_timestamp",
        to_timestamp(
            col("timestamp"),
            "yyyy-MM-dd HH:mm:ss"
        )
    )
    .withColumn(
        "ingestion_timestamp",
        current_timestamp()
    )
    .withColumn(
        "source_file",
        col("_metadata.file_path")
    )
    .drop("timestamp")
)

# Process only newly arrived files
bronze_query = (
    bronze_stream.writeStream
    .format("delta")
    .option(
        "checkpointLocation",
        checkpoint_path
    )
    .trigger(availableNow=True)
    .toTable(
        "fraud_detection.bronze.realtime_transactions"
    )
)

bronze_query.awaitTermination()

# Verification
bronze_count = spark.table(
    "fraud_detection.bronze.realtime_transactions"
).count()

print("Bronze ingestion completed")
print("Total Bronze records:", bronze_count)